In [1]:
# ============================================================
# MultiScaleSleepNet-Inspired Architecture + SupCon Loss +
# Artifact-Aware Weighting (merged: architecture-only ablation
# + your proven SupCon/artifact pipeline)
#
# What this combines:
#   - MultiScaleSleepNet-inspired arch (multi-scale CNN + FFT
#     spectral branch + SE + BiLSTM + Transformer) from your
#     architecture ablation run
#   - SupCon projection head + SupConLoss (Khosla et al. 2020)
#     from your context=7 SupCon+artifact-aware script
#   - Artifact-aware per-epoch loss weighting from TSV quality
#     annotations
#   - Honest 3-way split: checkpoint selection by VAL F1, test
#     evaluated exactly once per seed with the val-selected
#     checkpoint (same protocol as your architecture-only run)
#
# NOTE: your earlier script's header comment said "λ = 0.1" but
# the code actually used SUPCON_LAMBDA = 0.2. This script keeps
# 0.2 (matches what you actually ran + your thesis notes). Fix
# the stale comment in your own copy if you keep it around.
# ============================================================

import os
import csv
import math
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix
warnings.filterwarnings('ignore')

# ============================================================
# PATHS & CONFIG
# ============================================================
DATA_PATH     = r"D:\22\AA\preprocess\preprocessed_FFinal"
SPLIT_PATH    = r"D:\22\AA\AA journal\preprocess\preprocessed_split_v2"
ARTIFACT_PATH = r"D:\22\thesis 3233\Wearanize+_PlugNPlay_v1.0\Wearanize+_PlugNPlay_v1.0\derivatives\eegfloss-v1.0"
EVAL_PATH     = r"D:\22\AA\AA journal\evaluation\multiscale_supcon_artifact_c7_valfixed"
os.makedirs(EVAL_PATH, exist_ok=True)

LABEL_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
CONTEXT     = 7
WINDOW      = 2 * CONTEXT + 1     # 15
N_EPOCHS    = 30
BATCH_SIZE  = 64
SEEDS       = [42, 123, 256, 789, 999]

D_MODEL = 128
DROPOUT = 0.4
N_HEADS = 4

# SupCon hyperparameters (matches what you actually ran, not the stale comment)
SUPCON_LAMBDA = 0.2
SUPCON_TEMP   = 0.05
PROJ_DIM      = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device       : {device}")
print(f"Architecture : MultiScaleSleepNet-inspired (CNN+SE+BiLSTM+Transformer) + SupCon head")
print(f"Loss         : artifact-weighted CE + λ*SupCon (λ={SUPCON_LAMBDA}, τ={SUPCON_TEMP})")
print(f"Context      : {CONTEXT} (window={WINDOW})")
print(f"Output       : {EVAL_PATH}")

# ============================================================
# ARTIFACT WEIGHT CONFIG
# ============================================================
ARTIFACT_WEIGHT_MAP = {0: 1.0, 1: 0.0, 2: 0.4, 3: 0.2, 4: 0.1}
TARGET_CHANNELS   = ["PSG_F3:A2", "PSG_C3:A2"]
FALLBACK_CHANNELS = ["PSG_F3", "PSG_C3", "Zmax_EEGL", "Zmax_EEGR"]


def load_artifact_weights(subject_id, artifact_path, epoch_duration=30):
    tsv_path = os.path.join(
        artifact_path, subject_id, "eeg",
        f"{subject_id}_task-sleep_proc-artifacts.tsv"
    )
    if not os.path.exists(tsv_path):
        return None
    try:
        df = pd.read_csv(tsv_path, sep='\t')
    except Exception:
        return None

    available = [ch for ch in TARGET_CHANNELS if ch in df.columns]
    if not available:
        available = [ch for ch in FALLBACK_CHANNELS if ch in df.columns]
    if not available:
        return None

    df['worst_score'] = df[available].max(axis=1).astype(int)
    df['sleep_epoch'] = ((df['offset'] - 1e-6) // epoch_duration).astype(int)
    epoch_scores = df.groupby('sleep_epoch')['worst_score'].max()

    n_ep    = int(epoch_scores.index.max()) + 1
    weights = np.ones(n_ep, dtype=np.float32)
    for ep_idx, score in epoch_scores.items():
        weights[int(ep_idx)] = ARTIFACT_WEIGHT_MAP.get(int(score), 1.0)
    return weights


# ============================================================
# FIXED 3-WAY SPLIT (train / val / test — from make_train_val_split.py)
# ============================================================
_train_path = os.path.join(SPLIT_PATH, "_train_subs.npy")
_val_path   = os.path.join(SPLIT_PATH, "_val_subs.npy")
_test_path  = os.path.join(SPLIT_PATH, "_test_subs.npy")

for p, name in [(_train_path, "_train_subs.npy"),
                (_val_path,   "_val_subs.npy"),
                (_test_path,  "_test_subs.npy")]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"{name} not found at {SPLIT_PATH}. Run make_train_val_split.py first."
        )

TRAIN_SUBS = np.load(_train_path, allow_pickle=True).tolist()
VAL_SUBS   = np.load(_val_path,   allow_pickle=True).tolist()
TEST_SUBS  = np.load(_test_path,  allow_pickle=True).tolist()

assert set(TRAIN_SUBS).isdisjoint(VAL_SUBS),  "TRAIN/VAL subject overlap!"
assert set(TRAIN_SUBS).isdisjoint(TEST_SUBS), "TRAIN/TEST subject overlap!"
assert set(VAL_SUBS).isdisjoint(TEST_SUBS),   "VAL/TEST subject overlap!"

print(f"Split loaded -> Train:{len(TRAIN_SUBS)}  Val:{len(VAL_SUBS)}  Test:{len(TEST_SUBS)}")


# ============================================================
# DATASET — windowed epochs + per-epoch artifact weight
# ============================================================
class ArtifactAwareDataset(Dataset):
    def __init__(self, subject_list, data_path, artifact_path, context=CONTEXT):
        self.context = context
        self.data  = []
        self.index = []

        label_counter  = Counter()
        weight_counter = Counter()
        no_tsv_count   = 0

        for sub in subject_list:
            fp = os.path.join(data_path, f"{sub}.npz")
            if not os.path.exists(fp):
                continue
            with np.load(fp) as d:
                eeg    = d['eeg'][:, [0, 1], :]
                eog    = d['eog'][:, [0], :]
                signal = np.concatenate([eeg, eog], axis=1).astype(np.float32)
                labels = d['labels'].copy()

            n = len(labels)
            art_w = load_artifact_weights(sub, artifact_path)
            if art_w is None:
                art_w = np.ones(n, dtype=np.float32)
                no_tsv_count += 1
            else:
                if len(art_w) >= n:
                    art_w = art_w[:n]
                else:
                    pad = np.ones(n - len(art_w), dtype=np.float32)
                    art_w = np.concatenate([art_w, pad])

            sub_idx = len(self.data)
            self.data.append((signal, labels))
            for i in range(n):
                w = float(art_w[i])
                self.index.append((sub_idx, i, n, w))
                label_counter[int(labels[i])] += 1
                weight_counter[round(w, 1)] += 1

        self.label_counts = np.array(
            [label_counter[i] for i in range(5)], dtype=np.float32
        )
        total = len(self.index)
        ram = sum(s.nbytes for s, _ in self.data) / 1e9
        print(f"  Subjects: {len(self.data)}   Samples: {total:,}   RAM: {ram:.2f} GB")
        print(f"  No TSV  : {no_tsv_count} subjects (weight=1.0 default)")
        for w in sorted(weight_counter.keys(), reverse=True):
            pct = weight_counter[w] / total * 100
            print(f"    w={w:.1f} : {weight_counter[w]:>7,} ({pct:.1f}%)")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, center_i, n, weight = self.index[idx]
        signal, labels = self.data[sub_idx]
        window_epochs = []
        for offset in range(-self.context, self.context + 1):
            ei = max(0, min(n - 1, center_i + offset))
            window_epochs.append(signal[ei])
        x = np.stack(window_epochs, axis=0)
        y = int(labels[center_i])
        return (
            torch.FloatTensor(x),
            torch.tensor(y, dtype=torch.long),
            torch.tensor(weight, dtype=torch.float32),
        )


print(f"\nLoading .npz signal data from: {DATA_PATH}")
print("Building datasets...")
print("Train:")
train_ds = ArtifactAwareDataset(TRAIN_SUBS, DATA_PATH, ARTIFACT_PATH)
print("Val:")
val_ds   = ArtifactAwareDataset(VAL_SUBS, DATA_PATH, ARTIFACT_PATH)
print("Test:")
test_ds  = ArtifactAwareDataset(TEST_SUBS, DATA_PATH, ARTIFACT_PATH)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    if len(ds) == 0:
        raise RuntimeError(f"{name}_ds has 0 samples! Check DATA_PATH.")
print("Datasets ready.")


# ============================================================
# MULTI-SCALE CNN + SPECTRAL (FFT) BRANCH + SE BLOCK
# (unchanged from your architecture-only ablation)
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class MultiScaleCNN(nn.Module):
    def __init__(self, in_ch=3, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        mid = d_model // 4

        def time_branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=6, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(),
                nn.MaxPool1d(4, 4),
                nn.Conv1d(mid, mid, kernel_size=8, padding=4),
                nn.BatchNorm1d(mid), nn.GELU(),
                nn.MaxPool1d(2, 2),
                nn.Dropout(dropout),
            )

        self.small  = time_branch(25)
        self.medium = time_branch(50)
        self.large  = time_branch(100)

        self.spectral = nn.Sequential(
            nn.Conv1d(in_ch, mid, kernel_size=25, stride=6, padding=12),
            nn.BatchNorm1d(mid), nn.GELU(),
            nn.MaxPool1d(4, 4),
            nn.Conv1d(mid, mid, kernel_size=8, padding=4),
            nn.BatchNorm1d(mid), nn.GELU(),
            nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 3000)
            L_s = self.small(dummy).shape[2]
            L_m = self.medium(dummy).shape[2]
            L_l = self.large(dummy).shape[2]
            L_f = self.spectral(dummy).shape[2]

        target_L = min(L_s, L_m, L_l, L_f)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_m = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.pool_f = nn.AdaptiveAvgPool1d(target_L)
        self.out_len = target_L

        self.se = SEBlock(4 * mid)
        self.proj_cnn = nn.Sequential(
            nn.Conv1d(4 * mid, d_model, kernel_size=1),
            nn.BatchNorm1d(d_model), nn.GELU(),
        )

    def forward(self, x):
        fs = self.pool_s(self.small(x))
        fm = self.pool_m(self.medium(x))
        fl = self.pool_l(self.large(x))

        x_fft = torch.fft.rfft(x, dim=-1)
        x_mag = torch.abs(x_fft)
        if x_mag.shape[-1] < x.shape[-1]:
            pad = x.shape[-1] - x_mag.shape[-1]
            x_mag = F.pad(x_mag, (0, pad))
        else:
            x_mag = x_mag[..., :x.shape[-1]]
        ff = self.pool_f(self.spectral(x_mag))

        feat = torch.cat([fs, fm, fl, ff], dim=1)
        feat = self.se(feat)
        return self.proj_cnn(feat)


class BiLSTMTransformerBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, dropout=DROPOUT):
        super().__init__()
        self.bilstm = nn.LSTM(
            d_model, d_model // 2, num_layers=1,
            batch_first=True, bidirectional=True
        )
        self.norm1 = nn.LayerNorm(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 2, dropout=dropout,
            batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        lstm_out, _ = self.bilstm(x)
        x = self.norm1(x + lstm_out)
        attn_out = self.transformer(x)
        return self.norm2(x + attn_out)


# ============================================================
# FULL MODEL — MultiScaleSleepNet + SupCon projection head
# forward() now returns (logits, proj) like your SupCon script
# ============================================================
class MultiScaleSleepNetSupCon(nn.Module):
    def __init__(
        self, in_ch=3, d_model=D_MODEL, n_layers=2,
        dropout=DROPOUT, n_classes=5, context=CONTEXT, proj_dim=PROJ_DIM
    ):
        super().__init__()
        self.context = context
        self.d_model = d_model

        self.cnn = MultiScaleCNN(in_ch=in_ch, d_model=d_model, dropout=dropout)

        self.intra_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout)
            for _ in range(n_layers)
        ])

        self.inter_pos = nn.Parameter(torch.randn(1, WINDOW, d_model) * 0.01)
        self.inter_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout)
            for _ in range(n_layers)
        ])

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

        # SupCon projection head (train-only in effect; L2-normalized output)
        self.projector = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, proj_dim),
        )

    def forward(self, x):
        B, W, C, T = x.shape
        cnn_out = self.cnn(x.view(B * W, C, T))
        cnn_out = cnn_out.permute(0, 2, 1)
        intra = self.intra_blocks(cnn_out)
        epoch_feat = intra.mean(dim=1).view(B, W, self.d_model)

        inter = epoch_feat + self.inter_pos
        inter = self.inter_blocks(inter)
        center = inter[:, self.context, :]

        logits = self.classifier(center)
        proj = F.normalize(self.projector(center), dim=1)
        return logits, proj


# ============================================================
# SUPERVISED CONTRASTIVE LOSS (Khosla et al., 2020)
# ============================================================
class SupConLoss(nn.Module):
    def __init__(self, temperature=SUPCON_TEMP, mask_n1_only=False):
        super().__init__()
        self.temperature = temperature
        self.mask_n1_only = mask_n1_only

    def forward(self, features, labels):
        B = features.shape[0]
        dev = features.device

        sim = torch.matmul(features, features.T) / self.temperature
        sim_max, _ = torch.max(sim, dim=1, keepdim=True)
        sim = sim - sim_max.detach()

        labels_row = labels.unsqueeze(1)
        labels_col = labels.unsqueeze(0)
        pos_mask = (labels_row == labels_col).float()
        self_mask = torch.eye(B, device=dev)
        pos_mask = pos_mask - self_mask

        denom_mask = 1 - self_mask
        exp_sim = torch.exp(sim) * denom_mask
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)

        n_pos = pos_mask.sum(dim=1)
        if self.mask_n1_only:
            anchor_mask = (labels == 1).float()
        else:
            anchor_mask = (n_pos > 0).float()

        loss_per_sample = -(pos_mask * log_prob).sum(dim=1) / (n_pos + 1e-8)
        loss = (loss_per_sample * anchor_mask).sum() / (anchor_mask.sum() + 1e-8)
        return loss


# ============================================================
# ARTIFACT-WEIGHTED CE + COMBINED LOSS
# ============================================================
def artifact_weighted_ce(logits, labels, weights, class_weights):
    ce_loss = nn.CrossEntropyLoss(weight=class_weights, reduction='none')(logits, labels)
    weighted = ce_loss * weights
    valid = weights > 0
    if valid.sum() == 0:
        return weighted.mean()
    return weighted[valid].mean()


supcon_criterion = SupConLoss(temperature=SUPCON_TEMP, mask_n1_only=False)


def combined_loss(logits, proj, labels, weights, class_weights):
    l_ce = artifact_weighted_ce(logits, labels, weights, class_weights)
    l_supcon = supcon_criterion(proj, labels)
    return l_ce + SUPCON_LAMBDA * l_supcon, l_ce.item(), l_supcon.item()


# ============================================================
# CLASS WEIGHTS -- computed from TRAIN ONLY
# ============================================================
cw_np = train_ds.label_counts.sum() / (5 * train_ds.label_counts)
cw = torch.FloatTensor(cw_np).to(device)
print(f"\nClass weights (from TRAIN set only):")
for name, w in zip(LABEL_NAMES, cw_np):
    print(f"  {name}: {w:.3f}")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_epoch_fn(model, loader, optimizer, scheduler):
    model.train()
    total_loss = total_ce = total_sc = 0
    preds, labs = [], []
    for x, y, w in loader:
        x, y, w = x.to(device), y.to(device), w.to(device)
        optimizer.zero_grad()
        logits, proj = model(x)
        loss, l_ce, l_sc = combined_loss(logits, proj, y, w, cw)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        total_ce += l_ce
        total_sc += l_sc
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
    n = len(loader)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    return total_loss / n, total_ce / n, total_sc / n, acc, f1


@torch.no_grad()
def evaluate_fn(model, loader, need_clean_noisy=False):
    """
    Plain (unweighted) classification metrics -- used for VAL
    checkpoint selection every epoch, and for the final honest
    TEST evaluation. If need_clean_noisy=True (test set only),
    also splits accuracy by artifact weight (clean=1.0 vs noisy
    0<w<1) as a diagnostic, matching your SupCon+artifact script.
    """
    model.eval()
    preds, labs, weights_all = [], [], []
    for x, y, w in loader:
        x = x.to(device)
        logits, _ = model(x)   # proj not needed at eval
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.numpy())
        weights_all.extend(w.numpy())

    preds = np.array(preds); labs = np.array(labs); weights_all = np.array(weights_all)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labs, preds)
    per_cls = f1_score(labs, preds, average=None, zero_division=0, labels=[0, 1, 2, 3, 4])

    if not need_clean_noisy:
        return acc, f1, kappa, per_cls, preds, labs, None, None, 0, 0

    clean_mask = weights_all == 1.0
    noisy_mask = (weights_all > 0) & (weights_all < 1.0)
    clean_acc = accuracy_score(labs[clean_mask], preds[clean_mask]) if clean_mask.sum() > 0 else 0.0
    noisy_acc = accuracy_score(labs[noisy_mask], preds[noisy_mask]) if noisy_mask.sum() > 0 else 0.0
    return (acc, f1, kappa, per_cls, preds, labs,
            clean_acc, noisy_acc, int(clean_mask.sum()), int(noisy_mask.sum()))


# ============================================================
# CSV SETUP
# ============================================================
epoch_csv_path = os.path.join(EVAL_PATH, "epoch_log.csv")
epoch_fields = [
    "seed", "epoch",
    "train_loss", "train_ce", "train_supcon", "train_acc", "train_f1_macro",
    "val_acc", "val_f1_macro", "val_kappa",
    "val_f1_Wake", "val_f1_N1", "val_f1_N2", "val_f1_N3", "val_f1_REM",
    "lr", "is_best",
]
with open(epoch_csv_path, 'w', newline='') as f:
    csv.DictWriter(f, epoch_fields).writeheader()

csv_summary_path = os.path.join(EVAL_PATH, "summary.csv")
summary_fields = [
    "seed", "best_epoch", "best_val_f1",
    "test_acc", "test_f1_macro", "test_kappa",
    "test_f1_Wake", "test_f1_N1", "test_f1_N2", "test_f1_N3", "test_f1_REM",
    "clean_acc", "noisy_acc",
]
with open(csv_summary_path, 'w', newline='') as f:
    csv.DictWriter(f, summary_fields).writeheader()

print(f"\nPer-epoch log will be saved to : {epoch_csv_path}")
print(f"Final test summary saved to    : {csv_summary_path}")


# ============================================================
# 5-SEED TRAINING LOOP
# ============================================================
all_results = []

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSEED {seed}  ({seed_idx+1}/{len(SEEDS)})\n{'='*60}")
    set_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        generator=torch.Generator().manual_seed(seed)
    )
    val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = MultiScaleSleepNetSupCon(in_ch=3).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    proj_params = sum(p.numel() for p in model.projector.parameters())
    print(f"  Parameters : {n_params:,}  (projector: {proj_params:,})")

    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4,
                             betas=(0.9, 0.98), eps=1e-9)
    total_steps = N_EPOCHS * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    def warmup_cosine(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        t = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine)

    best_val_f1 = 0.0
    best_epoch = -1
    best_path = os.path.join(EVAL_PATH, f"best_seed{seed}.pt")

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, tr_ce, tr_sc, tr_acc, tr_f1 = train_epoch_fn(
            model, train_loader, optimizer, scheduler
        )

        # VAL -- unweighted metrics, used for checkpoint selection
        vl_acc, vl_f1, vl_kap, vl_per, _, _, _, _, _, _ = evaluate_fn(
            model, val_loader, need_clean_noisy=False
        )

        is_best = 0
        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_epoch = epoch
            torch.save(model.state_dict(), best_path)
            is_best = 1

        lr = optimizer.param_groups[0]['lr']
        tag = " <- BEST" if is_best else ""
        print(f"  Ep[{epoch:02d}/{N_EPOCHS}] "
              f"Loss:{tr_loss:.3f}(CE:{tr_ce:.3f}+SC:{tr_sc:.3f}) "
              f"TrAcc:{tr_acc:.3f} TrF1:{tr_f1:.3f} | "
              f"ValAcc:{vl_acc:.3f} ValF1:{vl_f1:.3f} Valk:{vl_kap:.3f} "
              f"LR:{lr:.2e}{tag}")

        with open(epoch_csv_path, 'a', newline='') as f:
            csv.DictWriter(f, epoch_fields).writerow({
                "seed": seed, "epoch": epoch,
                "train_loss": round(tr_loss, 6),
                "train_ce": round(tr_ce, 6),
                "train_supcon": round(tr_sc, 6),
                "train_acc": round(tr_acc, 6),
                "train_f1_macro": round(tr_f1, 6),
                "val_acc": round(vl_acc, 6),
                "val_f1_macro": round(vl_f1, 6),
                "val_kappa": round(vl_kap, 6),
                "val_f1_Wake": round(vl_per[0], 6),
                "val_f1_N1":   round(vl_per[1], 6),
                "val_f1_N2":   round(vl_per[2], 6),
                "val_f1_N3":   round(vl_per[3], 6),
                "val_f1_REM":  round(vl_per[4], 6),
                "lr": lr,
                "is_best": is_best,
            })

    # FINAL, ONE-TIME TEST EVALUATION (val-selected checkpoint)
    model.load_state_dict(torch.load(best_path, map_location=device))
    (test_acc, test_f1, test_kap, test_per, _, _,
     clean_acc, noisy_acc, n_clean, n_noisy) = evaluate_fn(
        model, test_loader, need_clean_noisy=True
    )

    print(f"\n  Seed {seed}: best val F1={best_val_f1:.4f} at epoch {best_epoch}")
    print(f"  Seed {seed} FINAL TEST (evaluated once): "
          f"Acc={test_acc*100:.2f}% F1={test_f1:.4f} k={test_kap:.4f}")
    print(f"  Clean Acc: {clean_acc*100:.2f}% ({n_clean})  Noisy Acc: {noisy_acc*100:.2f}% ({n_noisy})")
    for i, name in enumerate(LABEL_NAMES):
        print(f"    {name:6s}: {test_per[i]:.4f}")

    all_results.append({
        'seed': seed, 'acc': test_acc, 'f1': test_f1, 'kappa': test_kap,
        'per_cls': test_per, 'best_epoch': best_epoch, 'best_val_f1': best_val_f1,
        'clean_acc': clean_acc, 'noisy_acc': noisy_acc,
    })

    with open(csv_summary_path, 'a', newline='') as f:
        csv.DictWriter(f, summary_fields).writerow({
            "seed": seed,
            "best_epoch": best_epoch,
            "best_val_f1": round(best_val_f1, 4),
            "test_acc": round(test_acc, 4),
            "test_f1_macro": round(test_f1, 4),
            "test_kappa": round(test_kap, 4),
            "test_f1_Wake": round(test_per[0], 4),
            "test_f1_N1":   round(test_per[1], 4),
            "test_f1_N2":   round(test_per[2], 4),
            "test_f1_N3":   round(test_per[3], 4),
            "test_f1_REM":  round(test_per[4], 4),
            "clean_acc": round(clean_acc, 4),
            "noisy_acc": round(noisy_acc, 4),
        })

# ============================================================
# FINAL REPORT
# ============================================================
accs = np.array([r['acc'] for r in all_results]) * 100
f1s = np.array([r['f1'] for r in all_results])
kappas = np.array([r['kappa'] for r in all_results])
n1s = np.array([r['per_cls'][1] for r in all_results])

print(f"\n{'='*60}\nMULTISCALESLEEPNET + SUPCON + ARTIFACT-AWARE (VAL-SELECTED, HONEST TEST) -- 5-SEED REPORT\n{'='*60}")
print(f"Accuracy : {accs.mean():.2f} +- {accs.std():.2f}%")
print(f"Macro F1 : {f1s.mean():.4f} +- {f1s.std():.4f}")
print(f"Kappa    : {kappas.mean():.4f} +- {kappas.std():.4f}")
print(f"N1 F1    : {n1s.mean():.4f} +- {n1s.std():.4f}")

print(f"\nAll per-epoch train/val metrics : {epoch_csv_path}")
print(f"Final per-seed test summary     : {csv_summary_path}")
print("Done! Run this, then re-run your ensemble script pointed at this EVAL_PATH")
print("to see whether the ensembled Macro F1 clears 0.81.")

Device       : cuda
Architecture : MultiScaleSleepNet-inspired (CNN+SE+BiLSTM+Transformer) + SupCon head
Loss         : artifact-weighted CE + λ*SupCon (λ=0.2, τ=0.05)
Context      : 7 (window=15)
Output       : D:\22\AA\AA journal\evaluation\multiscale_supcon_artifact_c7_valfixed
Split loaded -> Train:65  Val:11  Test:20

Loading .npz signal data from: D:\22\AA\preprocess\preprocessed_FFinal
Building datasets...
Train:
  Subjects: 65   Samples: 60,605   RAM: 2.18 GB
  No TSV  : 0 subjects (weight=1.0 default)
    w=1.0 :  49,795 (82.2%)
    w=0.4 :     837 (1.4%)
    w=0.2 :   6,766 (11.2%)
    w=0.1 :   3,207 (5.3%)
Val:
  Subjects: 11   Samples: 10,742   RAM: 0.39 GB
  No TSV  : 0 subjects (weight=1.0 default)
    w=1.0 :   9,508 (88.5%)
    w=0.4 :      93 (0.9%)
    w=0.2 :     920 (8.6%)
    w=0.1 :     178 (1.7%)
    w=0.0 :      43 (0.4%)
Test:
  Subjects: 20   Samples: 19,763   RAM: 0.71 GB
  No TSV  : 0 subjects (weight=1.0 default)
    w=1.0 :  15,258 (77.2%)
    w=0.4 :    